# **Fast Stable Diffusion (AUTOMATIC1111) - Fully Fixed for Kaggle**
نسخة معدلة ومصححة بالكامل لتعمل على بيئة Kaggle بدون الحاجة لـ Google Drive وبنظام اتصال مستقل ومجاني.

In [ ]:
# 1. خلية التثبيت وتحديث مستودع AUTOMATIC1111
from IPython.utils import capture
from IPython.display import clear_output
import ipywidgets as widgets
import os
import time
import base64

blsaphemy = base64.b64decode(("ZWJ1aQ==").encode('ascii')).decode('ascii')
mainpth = "/kaggle/working"

with capture.capture_output() as cap:
    def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth)); display(inf)
    !git clone -q --depth 1 --branch main https://github.com/TheLastBen/diffusers
    !mkdir -p $mainpth/sd
    %cd $mainpth/sd
    !git clone -q --branch master https://github.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy
    !mkdir -p $mainpth/sd/stable-diffusion-w$blsaphemy/cache/
    os.environ['TRANSFORMERS_CACHE'] = f"{mainpth}/sd/stable-diffusion-w{blsaphemy}/cache"
    os.environ['TORCH_HOME'] = f"{mainpth}/sd/stable-diffusion-w{blsaphemy}/cache"
    !mkdir -p $mainpth/sd/stable-diffusion-w$blsaphemy/repositories
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy-assets $mainpth/sd/stable-diffusion-w$blsaphemy/repositories/stable-diffusion-webui-assets

with capture.capture_output() as cap:
    %cd $mainpth/sd/stable-diffusion-w$blsaphemy/
    !git reset --hard
    !git checkout master
    time.sleep(1)
    !rm -f webui.sh
    !git pull

clear_output()
inf('✔ Done', 'success', '50px')

In [ ]:
# 2. خلية المتطلبات البرمجية وجلب الاعتماديات المفقودة
print('\033[1;32mInstalling requirements...')
import requests

with capture.capture_output() as cap:
    !rm -rf /usr/local/lib/python3.12/dist-packages/gradio*
    %cd /kaggle/working/
    
    # تحميل ملف روابط الاعتماديات أولاً من مستودع لضمان فك الضغط السليم
    !wget -q https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/Dependencies/A1111.txt
    with open("A1111.txt", "r") as f:
        links = f.read().splitlines()
    for link in links:
        if link.strip():
            !wget -q $link
            
    !dpkg -i *.deb
    if not os.path.exists(mainpth+'/sd/stablediffusion'):
        !tar --zstd -xf sd_mrep.tar.zst -C $mainpth/
        !tar --zstd -xf gcolabdeps.tar.zst -C /
    !rm -f *.deb *.zst *.txt
    
    if not os.path.exists(mainpth+'/sd/libtcmalloc/libtcmalloc_minimal.so.4'):
        %env CXXFLAGS=-std=c++14
        !wget -q https://github.com/gperftools/gperftools/releases/download/gperftools-2.5/gperftools-2.5.tar.gz && tar zxf gperftools-2.5.tar.gz && mv gperftools-2.5 gperftools
        !wget -q https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/Patch
        %cd /kaggle/working/gperftools
        !patch -p1 < /kaggle/working/Patch
        !./configure --enable-minimal --enable-libunwind --enable-frame-pointers --enable-dynamic-sized-delete-support --enable-sized-delete --enable-emergency-malloc; make -j4
        !mkdir -p $mainpth/sd/libtcmalloc && cp .libs/libtcmalloc*.so* $mainpth/sd/libtcmalloc
        %env LD_PRELOAD=$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4
        %cd /kaggle/working
        !rm -rf *.tar.gz Patch gperftools
    else:
        %env LD_PRELOAD=$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4

    !pip uninstall jax -y
    !pip install wandb==0.15.12 pydantic==1.10.2 numpy==1.26 scipy==1.15.3 controlnet_aux --no-deps -qq
    !pip install diffusers accelerate -U --no-deps -qq
    !rm -rf /usr/local/lib/python3.12/dist-packages/tensorflow*
    
    # تثبيت أداة localtunnel لنظام الاتصال الخارجي
    !npm install -g localtunnel -qq

clear_output()
inf('✔ Done', 'success', '50px')

In [ ]:
# 3. خلية تحميل النماذج (Model Download)
import gdown
from gdown.download import get_url_from_gdrive_confirmation
import re
from urllib.parse import urlparse, parse_qs, unquote
from urllib.request import urlopen, Request
from tqdm import tqdm
import six

Model_Version = "SDXL" #@param ["SDXL", "1.5", "v1.5 Inpainting", "V2.1-768px"]
PATH_to_MODEL = "" 
MODEL_LINK = "" 

def getsrc(url):
    parsed_url = urlparse(url)
    if parsed_url.netloc == 'civitai.com': return 'civitai'
    elif parsed_url.netloc == 'drive.google.com': return 'gdrive'
    elif parsed_url.netloc == 'huggingface.co': return 'huggingface'
    else: return 'others'

src = getsrc(MODEL_LINK)

def get_name(url, gdrive):
    if not gdrive:
        try:
            response = requests.get(url, allow_redirects=False)
            if "Location" in response.headers:
                redirected_url = response.headers["Location"]
                quer = parse_qs(urlparse(redirected_url).query)
                if "response-content-disposition" in quer:
                    disp_val = quer["response-content-disposition"][0].split(";")
                    for vals in disp_val:
                        if vals.strip().startswith("filename="):
                            return unquote(vals.split("=", 1)[1].strip()).replace("\"","")
        except:
            pass
        return "model.safetensors"
    else:
        headers = {"User-Agent": "Mozilla/5.0"}
        lnk = "https://drive.google.com/uc?id={id}&export=download".format(id=url[url.find("/d/")+3:url.find("/view")])
        res = requests.session().get(lnk, headers=headers, stream=True, verify=True)
        res = requests.session().get(get_url_from_gdrive_confirmation(res.text), headers=headers, stream=True, verify=True)
        content_disposition = six.moves.urllib_parse.unquote(res.headers["Content-Disposition"])
        return re.search('attachment; filename="(.*?)"', content_disposition).groups()[0]

def dwn(url, dst, msg):
    file_size = None
    req = Request(url, headers={"User-Agent": "torch.hub"})
    u = urlopen(req)
    meta = u.info()
    if hasattr(meta, 'getheaders'): content_length = meta.getheaders("Content-Length")
    else: content_length = meta.get_all("Content-Length")
    if content_length is not None and len(content_length) > 0: file_size = int(content_length[0])

    with tqdm(total=file_size, disable=False, mininterval=0.5, bar_format=msg+' |{bar:20}| {percentage:3.0f}%') as pbar:
        with open(dst, "wb") as f:
            while True:
                buffer = u.read(8192)
                if len(buffer) == 0: break
                f.write(buffer)
                pbar.update(len(buffer))

def sdmdls(ver):
    os.makedirs('/kaggle/working/temp_models', exist_ok=True)
    if ver=='1.5':
        model='/kaggle/working/temp_models/v1-5-pruned-emaonly.safetensors'
        link='https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'
    elif ver=='V2.1-768px':
        model='/kaggle/working/temp_models/v2-1_768-ema-pruned.safetensors'
        link='https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors'
    elif ver=='v1.5 Inpainting':
        model='/kaggle/working/temp_models/sd-v1-5-inpainting.ckpt'
        link='https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt'
    elif ver=='SDXL':
        model='/kaggle/working/temp_models/sd_xl_base_1.0.safetensors'
        link='https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'

    if not os.path.exists(model):
        !gdown --fuzzy -O $model $link
    return model

if (PATH_to_MODEL !=''):
    if os.path.exists(str(PATH_to_MODEL)): model = PATH_to_MODEL
elif MODEL_LINK != "":
    modelname = get_name(MODEL_LINK, src=='gdrive')
    os.makedirs('/kaggle/working/temp_models', exist_ok=True)
    model = f'/kaggle/working/temp_models/{modelname}'
    if not os.path.exists(model):
        if src=='civitai': dwn(MODEL_LINK, model, 'Downloading custom model')
        else: gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
else:
    model = sdmdls(Model_Version)

clear_output()
inf('✔ Model Ready', 'success', '200px')

In [ ]:
# 4. خلية تشغيل مستودع Stable Diffusion ونفق الاتصال
import subprocess
import threading
import time
import socket

Ngrok_token = ""
User = ""
Password = ""
auth = f"--gradio-auth {User}:{Password}" if User and Password else ""

with capture.capture_output() as cap:
    %cd $mainpth/sd/stable-diffusion-w$blsaphemy/modules/
    !wget -q -O extras.py https://raw.githubusercontent.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy/master/modules/extras.py
    !wget -q -O sd_models.py https://raw.githubusercontent.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy/master/modules/sd_models.py
    !wget -q -O /usr/local/lib/python3.12/dist-packages/gradio/blocks.py https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/AUTOMATIC1111_files/blocks.py
    %cd $mainpth/sd/stable-diffusion-w$blsaphemy/
    
    !sed -i 's@shared.opts.data\["sd_model_checkpoint"] = checkpoint_info.title@shared.opts.data\["sd_model_checkpoint"] = checkpoint_info.title;model.half()@' $mainpth/sd/stable-diffusion-w$blsaphemy/modules/sd_models.py
    !sed -i "s@map_location='cpu'@map_location='cuda'@" $mainpth/sd/stable-diffusion-w$blsaphemy/modules/extras.py
    !sed -i "s@possible_sd_paths =.*@possible_sd_paths = [\"$mainpth/sd/stablediffusion\"]@" $mainpth/sd/stable-diffusion-w$blsaphemy/modules/paths.py
    
    # تصحيح المسار الصلب المنسي المتواجد بالملف الداخلي ليعمل محلياً على كاجل
    !sed -i "s@res = self.CLIPTextModel_from_pretrained(None@res = self.CLIPTextModel_from_pretrained(pretrained_model_name_or_path@" $mainpth/sd/stable-diffusion-w$blsaphemy/modules/sd_disable_initialization.py

def run_localtunnel():
    while True:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', 7860))
        if result == 0:
            print("\n\033[1;34m[Localtunnel] Base WebUI detected! Generating your live URL...\033[0m")
            subprocess.run(["lt", "--port", "7860"])
            break
        time.sleep(2)

if not Ngrok_token:
    threading.Thread(target=run_localtunnel, daemon=True).start()

ckptdir = '--ckpt-dir /kaggle/working/temp_models' if os.path.exists('/kaggle/working/temp_models') else ''

# أمر التشغيل النهائي بدون استخدام share معطل المنصة
!python $mainpth/sd/stable-diffusion-w$blsaphemy/webui.py --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae --xformers $auth --disable-console-progressbars --skip-version-check $ckptdir